In [ ]:
"""
Generic Google Earth Engine ML Pipeline Template
------------------------------------------------
This script demonstrates:
- Sentinel-2 preprocessing (cloud masking & compositing)
- Feature engineering
- Sampling raster values at point locations
- Training a Random Forest regression model
- Exporting predictions and feature importance

NOTE:
- Replace placeholder asset paths with own data
- Thresholds and weighting strategies are illustrative only
"""

import ee

# ============================================================================
# USER SETTINGS
# ============================================================================

FILENAME = "example_ml_pipeline"
TARGET_PROPERTY = 'target'   # Replace with your variable
EXPORT_FOLDER = 'GEE_exports'

USE_LOG_TRANSFORM = False
USE_OVERSAMPLING = False

SEED = 42

# ============================================================================
# INITIALIZE EARTH ENGINE
# ============================================================================

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("Earth Engine initialized")

# ============================================================================
# LOAD DATA (PLACEHOLDERS)
# ============================================================================

# Replace with your own assets
points = ee.FeatureCollection("YOUR_POINT_ASSET")
region = ee.Geometry.Rectangle([-25, 35, 45, 72])  # Example region

points = points.filterBounds(region)

# ============================================================================
# CREATE FEATURE IMAGE
# ============================================================================

def create_feature_image():

    START_DATE = "2019-01-01"
    END_DATE = "2021-01-01"

    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(region)
          .filterDate(START_DATE, END_DATE)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
          .select(['B2','B3','B4','B8']))

    # Simple cloud mask (generic)
    def mask_clouds(img):
        return img.updateMask(img.select('B2').lt(3000))

    s2 = s2.map(mask_clouds)

    # Median composite
    composite = s2.median()

    # Basic indices
    ndvi = composite.normalizedDifference(['B8','B4']).rename('NDVI')
    ndwi = composite.normalizedDifference(['B3','B8']).rename('NDWI')

    # Example custom features (genericized)
    b2 = composite.select('B2').multiply(0.0001)
    b4 = composite.select('B4').multiply(0.0001)

    custom_index_1 = b4.subtract(b2).rename('custom_index_1')
    custom_index_2 = b4.pow(2).add(b2.pow(2)).sqrt().rename('custom_index_2')

    # Environmental layers (public datasets)
    elevation = ee.Image('USGS/SRTMGL1_003').rename('elevation')
    worldclim = ee.Image('WORLDCLIM/V1/BIO')
    temperature = worldclim.select('bio01').rename('temperature')

    features = (ndvi
                .addBands(ndwi)
                .addBands(custom_index_1)
                .addBands(custom_index_2)
                .addBands(elevation)
                .addBands(temperature))

    return features

feature_image = create_feature_image()
bands = feature_image.bandNames().getInfo()

print("Feature bands:", bands)

# ============================================================================
# SAMPLE DATA
# ============================================================================

sampled = feature_image.sampleRegions(
    collection=points,
    properties=[TARGET_PROPERTY],
    scale=30,
    tileScale=8
)

sampled = sampled.filter(ee.Filter.notNull([TARGET_PROPERTY]))

# Optional log transform
if USE_LOG_TRANSFORM:
    def add_log(f):
        val = ee.Number(f.get(TARGET_PROPERTY))
        return f.set('target_log', val.add(1).log())
    sampled = sampled.map(add_log)
    target = 'target_log'
else:
    target = TARGET_PROPERTY

# ============================================================================
# TRAIN / TEST SPLIT
# ============================================================================

def split_data(fc, seed=42):
    fc = fc.randomColumn('random', seed)
    train = fc.filter(ee.Filter.lt('random', 0.8))
    test  = fc.filter(ee.Filter.gte('random', 0.8))
    return train, test

train_fc, test_fc = split_data(sampled, SEED)

# ============================================================================
# MODEL TRAINING
# ============================================================================

classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    seed=SEED
).setOutputMode('REGRESSION')

model = classifier.train(
    features=train_fc,
    classProperty=target,
    inputProperties=bands
)

print("Model trained")

# ============================================================================
# PREDICTIONS
# ============================================================================

train_pred = train_fc.classify(model, 'prediction')
test_pred  = test_fc.classify(model, 'prediction')

# ============================================================================
# EXPORTS
# ============================================================================

def export_table(fc, name):
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=name,
        folder=EXPORT_FOLDER,
        fileFormat='CSV'
    )
    task.start()
    return task

def export_importance(model, name):
    importance = ee.FeatureCollection([ee.Feature(None, model.explain())])
    return export_table(importance, name)

export_table(train_pred, f"{FILENAME}_train")
export_table(test_pred,  f"{FILENAME}_test")
export_importance(model, f"{FILENAME}_importance")

print("Exports started. Check Earth Engine Tasks tab.")